In [1]:
from src.create_graphs import create_graph_list
# from src.load_data import load_data
from src.VSA_conversion import VSA_conversion
from src.embeddings import getEmbedding
from models.graphcnnVSA_Binding_FULL import GraphCNN
import torch
from xgboost import XGBRegressor
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from rdkit import Chem
from torch_geometric.data import InMemoryDataset, Data


In [ ]:
import numpy as np
import torch
from torch_geometric.data import Data
from rdkit import Chem
from rdkit.Chem import AllChem


# -----------------------------
# 1. Bond node features
# -----------------------------

def bond_node_features(bond, pos):
    """
    Features for each bond (node in the bond-angle graph).

    Returns a 1D numpy array, e.g.:
        [bond_type, is_conjugated, in_ring, bond_length]
    """
    bt = bond.GetBondType()
    bond_type = {
        Chem.rdchem.BondType.SINGLE: 1,
        Chem.rdchem.BondType.DOUBLE: 2,
        Chem.rdchem.BondType.TRIPLE: 3,
        Chem.rdchem.BondType.AROMATIC: 4,
    }.get(bt, 0)

    is_conjugated = int(bond.GetIsConjugated())
    in_ring = int(bond.IsInRing())

    a = bond.GetBeginAtomIdx()
    b = bond.GetEndAtomIdx()
    length = float(np.linalg.norm(pos[a] - pos[b]))  # 3D bond length

    return np.array([bond_type, is_conjugated, in_ring, length],
                    dtype=np.float32)


# -----------------------------
# 2. Main builder
# -----------------------------

def smiles_to_bond_angle_graph(smiles):
    """
    Build a bond-angle graph from a SMILES string.

    Returns a PyG Data object with fields:
      - data.bond_x              : [num_bonds, F_bond]  (node features)
      - data.angle_edge_index    : [2, num_angles*2]    (directed edges)
      - data.angle_edge_attr     : [num_angles*2, 3]    (edge features)

    Edge features are [theta, cos(theta), sin(theta)] where theta is
    the bond angle at the central atom, in radians.
    """
    # ----- RDKit molecule + 3D coords -----
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    mol = Chem.AddHs(mol)

    # Generate a 3D conformer
    params = AllChem.ETKDGv3()
    params.randomSeed = 0xf00d
    if AllChem.EmbedMolecule(mol, params) != 0:
        raise RuntimeError(f"3D embedding failed for: {smiles}")
    AllChem.MMFFOptimizeMolecule(mol)

    conf = mol.GetConformer()
    num_atoms = mol.GetNumAtoms()

    # Positions [N, 3]
    pos = np.zeros((num_atoms, 3), dtype=np.float32)
    for i in range(num_atoms):
        p = conf.GetAtomPosition(i)
        pos[i] = [p.x, p.y, p.z]

    # ----- Bonds and bond node features -----
    bonds = list(mol.GetBonds())
    num_bonds = len(bonds)
    if num_bonds == 0:
        raise RuntimeError(f"No bonds found in molecule: {smiles}")

    bond_x_list = []        # node features
    bond_endpoints = []     # (a, b) for each bond id

    for bond in bonds:
        a = bond.GetBeginAtomIdx()
        b = bond.GetEndAtomIdx()
        bond_endpoints.append((a, b))
        bond_x_list.append(bond_node_features(bond, pos))

    bond_x = torch.tensor(np.vstack(bond_x_list), dtype=torch.float32)  # [B, F_bond]

    # ----- Bond-angle edges: nodes = bonds, edges = angles -----

    # For each atom v, get incident bonds (bond_id, neighbor_atom)
    atom_to_bonds = [[] for _ in range(num_atoms)]
    for bond_id, (a, b) in enumerate(bond_endpoints):
        atom_to_bonds[a].append((bond_id, b))
        atom_to_bonds[b].append((bond_id, a))

    angle_edge_pairs = []   # list of [bond_i, bond_j]
    angle_feat_list  = []   # list of [theta, cos(theta), sin(theta)]

    for v in range(num_atoms):
        inc = atom_to_bonds[v]
        if len(inc) < 2:
            continue

        # All unordered pairs of incident bonds at atom v
        for i in range(len(inc)):
            bond_id1, u = inc[i]
            for j in range(i + 1, len(inc)):
                bond_id2, w = inc[j]

                # Angle u - v - w at atom v
                vec1 = pos[u] - pos[v]
                vec2 = pos[w] - pos[v]
                norm1 = np.linalg.norm(vec1)
                norm2 = np.linalg.norm(vec2)
                if norm1 < 1e-6 or norm2 < 1e-6:
                    continue

                cos_theta = float(np.dot(vec1, vec2) / (norm1 * norm2))
                cos_theta = float(np.clip(cos_theta, -1.0, 1.0))
                theta = float(np.arccos(cos_theta))  # radians

                angle_feat = np.array(
                    [theta, np.cos(theta), np.sin(theta)],
                    dtype=np.float32
                )

                # Add two directed edges for this angle: bond1→bond2 and bond2→bond1
                angle_edge_pairs.append([bond_id1, bond_id2])
                angle_feat_list.append(angle_feat)

                angle_edge_pairs.append([bond_id2, bond_id1])
                angle_feat_list.append(angle_feat)

    if len(angle_edge_pairs) > 0:
        angle_edge_index = torch.tensor(angle_edge_pairs, dtype=torch.long).t().contiguous()
        angle_edge_attr  = torch.tensor(np.vstack(angle_feat_list), dtype=torch.float32)
    else:
        # molecule with no angles (e.g. single bond)
        angle_edge_index = torch.empty((2, 0), dtype=torch.long)
        angle_edge_attr  = torch.empty((0, 3), dtype=torch.float32)

    # ----- Wrap in a PyG Data object -----
    data = Data()
    data.bond_x           = bond_x              # [num_bonds, F_bond] node features
    data.angle_edge_index = angle_edge_index    # [2, num_angle_edges]
    data.angle_edge_attr  = angle_edge_attr     # [num_angle_edges, 3]

    return data


In [5]:
class S2VGraph(object):
    def __init__(self,
                 g,
                 label,
                 mol,
                 node_tags=None,
                 node_features=None,
                 # new stuff for bond-angle graph H:
                 bond_x=None,              # [num_bonds, F_bond]  node features in H
                 bond_endpoints=None,      # list of (a, b) atom indices for each bond id
                 angle_edge_index=None,    # [2, num_angle_edges] (bond_i, bond_j)
                 angle_edge_attr=None):    # [num_angle_edges, F_angle]
        """
        g              : atom graph G (e.g. a networkx.Graph of atoms)
        label          : graph label (e.g. LogS bucket or regression target index)
        mol            : RDKit Mol (optional, for debugging / visualization)
        node_tags      : list of integer tags for atom nodes
        node_features  : torch.FloatTensor [num_atoms, F_atom] (input features)

        --- Bond-angle graph H (optional) ---
        bond_x         : torch.FloatTensor [num_bonds, F_bond] (bond node features)
        bond_endpoints : list or tensor of shape [num_bonds, 2] with atom indices (a,b)
        angle_edge_index : torch.LongTensor [2, num_angle_edges] (edges between bonds)
        angle_edge_attr  : torch.FloatTensor [num_angle_edges, F_angle] (edge features)
        """

        self.label = label
        self.g = g
        self.mol = mol

        # atom graph data
        self.node_tags = node_tags
        self.node_features = node_features
        self.neighbors = []      # you probably fill this later
        self.edge_mat = 0
        self.max_neighbor = 0

        # bond-angle graph data (H)
        self.bond_x = bond_x
        self.bond_endpoints = bond_endpoints
        self.angle_edge_index = angle_edge_index
        self.angle_edge_attr = angle_edge_attr

In [9]:
from rdkit import Chem

def build_atom_nx_graph(mol):
    import networkx as nx
    g = nx.Graph()
    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        g.add_node(i)
    for bond in mol.GetBonds():
        a = bond.GetBeginAtomIdx()
        b = bond.GetEndAtomIdx()
        g.add_edge(a, b)
    return g

def smiles_to_S2VGraph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)

    # 1) Atom graph G as networkx
    g_nx = build_atom_nx_graph(mol)

    # 2) Atom node features (whatever you already use)
    #    here just a dummy example:
    node_features = None
    node_tags = None

    # 3) Bond-angle graph H (bonds as nodes, angles as edges)
    ba = smiles_to_bond_angle_graph(smiles)   # from previous answer

    bond_x = ba.bond_x                       # [num_bonds, F_bond]
    # bond_endpoints = ba.bond_endpoints       # we can store (a,b) if you add it there
    bond_endpoints = None
    angle_edge_index = ba.angle_edge_index   # [2, E_angle]
    angle_edge_attr  = ba.angle_edge_attr    # [E_angle, F_angle]

    # 4) Wrap everything into ONE S2VGraph object
    g_obj = S2VGraph(
        g=g_nx,
        label=y,
        mol=mol,
        node_tags=node_tags,
        node_features=node_features,
        bond_x=bond_x,
        bond_endpoints=bond_endpoints,
        angle_edge_index=angle_edge_index,
        angle_edge_attr=angle_edge_attr,
    )

    return g_obj

In [11]:
import pandas as pd

class ZINCLikeCSV(InMemoryDataset):
    def __init__(self, csv_path, smiles_col="smiles_canon", target_col="LogS"):
        df = pd.read_csv(csv_path)
        super().__init__('.')
        graphs = []
        for smi, y in zip(df[smiles_col], df[target_col]):
            g = smiles_to_bond_angle_graph(smi, y)
            if g is not None:
                graphs.append(g)
        self.data, self.slices = self.collate(graphs)

def load_data():
  dataset_test  = ZINCLikeCSV("final_data/final_unique_test.csv")
  dataset_train  = ZINCLikeCSV("final_data/final_unique_train_fixed.csv")

  return dataset_train, dataset_test #, gl_train, 


train_data, test_data = load_data()

TypeError: smiles_to_bond_angle_graph() takes 1 positional argument but 2 were given